# BindingDB Kinase Data

**Author:** Benjamin Stickland  
**Affiliation:** UNC Chapel Hill
**Date:** 2026

In [5]:
import os
import re
import shutil
import zipfile
from pathlib import Path
import asyncio
import aiohttp
import requests
import pandas as pd
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from tqdm import tqdm
import nest_asyncio
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
nest_asyncio.apply()
RDLogger.DisableLog('rdApp.*')

AFFINITY_CUTOFF = 1000

# Input & Output Path
The BindingDB reference sets used analysis within the paper (October 2025 BindingDB release; UniProt October 2025; KLIFS) are provided in `Data/BindingDB_Kinase_Inhibitors.csv` and `Data/BindingDB_Non_Kinase_Inhibitors.csv`. Running this notebook writes new copies to `Output/Intermediate/Regenerated_Inputs/`

In [6]:
PROJECT_DIR = Path.cwd() if (Path.cwd() / 'Data').is_dir() else Path.cwd().parent
DATA_DIR = str(PROJECT_DIR / 'Data')
UNIPROT_FILE = os.path.join(DATA_DIR, 'uniprotkb_keyword_KW_0418_AND_reviewed_2025_10_22.xlsx')
OUTPUT_DIR = PROJECT_DIR / 'Output' / 'Intermediate' / 'Regenerated_Inputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_KINASE = str(OUTPUT_DIR / 'BindingDB_Kinase_Inhibitors.csv')
OUTPUT_NON_KINASE = str(OUTPUT_DIR / 'BindingDB_Non_Kinase_Inhibitors.csv')

# Load UniProt kinase list

In [7]:
df_uniprot = pd.read_excel(UNIPROT_FILE)
uniprot_ids = df_uniprot["Entry"].dropna().tolist()

print(f"Loaded {len(uniprot_ids)} UniProt kinase IDs")

Loaded 21232 UniProt kinase IDs


# Validate kinases via KLIFS API

In [8]:
async def query_klifs(session, uniprot_id):
    url = f"https://klifs.net/api/kinase_ID?kinase_name={uniprot_id}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            if isinstance(data, list) and data:
                return uniprot_id
    except:
        pass
    return None

async def validate_all_kinases(uniprot_ids):
    valid_kinases = set()
    timeout = aiohttp.ClientTimeout(total=600)
    connector = aiohttp.TCPConnector(limit=50)
    
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        tasks = [query_klifs(session, uid) for uid in uniprot_ids]
        results = []
        
        for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Validating kinases"):
            result = await task
            if result is not None:
                valid_kinases.add(result)
    
    return valid_kinases

try:
    kinase_ids = await validate_all_kinases(uniprot_ids)
except RuntimeError:
    loop = asyncio.get_event_loop()
    kinase_ids = loop.run_until_complete(validate_all_kinases(uniprot_ids))

failed = len(uniprot_ids) - len(kinase_ids)
print(f"Validated {len(kinase_ids)} kinases ({failed} failed)")

Validating kinases: 100%|██████████| 21232/21232 [04:30<00:00, 78.47it/s] 

Validated 979 kinases (20253 failed)


# Fetch BindingDB

The paper used the October 2025 release, which BindingDB no longer offers for download. A newer release can change these results slightly. To reproduce the paper, use the provided reference sets described above, which notebook 03 reads by default.

In [9]:
BINDINGDB_DOWNLOADS = 'https://www.bindingdb.org/rwd/bind/chemsearch/marvin/Download.jsp'
BINDINGDB_URL = 'https://www.bindingdb.org/rwd/bind/downloads/{}'

def get_bindingdb(data_dir):
    """Use a local BindingDB TSV or download the current release."""
    data_dir = Path(data_dir)
    local_files = sorted(data_dir.glob('BindingDB_All_*.tsv'))
    if local_files:
        return local_files[-1]

    response = requests.get(BINDINGDB_DOWNLOADS, timeout=120)
    response.raise_for_status()
    releases = re.findall(r'BindingDB_All_\d{6}_tsv\.zip', response.text)
    if not releases:
        raise RuntimeError('Could not find the current BindingDB TSV on the download page.')
    filename = max(releases)
    zip_path = data_dir / filename
    tsv_path = data_dir / filename.replace('_tsv.zip', '.tsv')

    with requests.get(BINDINGDB_URL.format(filename), stream=True, timeout=120) as response:
        response.raise_for_status()
        size = int(response.headers.get('content-length', 0))
        with open(zip_path, 'wb') as output, tqdm(total=size, unit='B', unit_scale=True, desc=filename) as bar:
            for chunk in response.iter_content(chunk_size=2 ** 20):
                output.write(chunk)
                bar.update(len(chunk))

    with zipfile.ZipFile(zip_path) as archive:
        member = next(name for name in archive.namelist() if name.endswith('.tsv'))
        with archive.open(member) as source, open(tsv_path, 'wb') as output:
            shutil.copyfileobj(source, output)
    zip_path.unlink()
    return tsv_path

BINDINGDB_FILE = str(get_bindingdb(DATA_DIR))
print(f'Using {BINDINGDB_FILE}')

Using c:\Users\bstrickland\Documents\Projects\2026_KIPS_Final_Revisions\Code\Data\BindingDB_All_202510.tsv


# Load BindingDB

In [10]:
print("\nLoading BindingDB TSV...")
df = pd.read_csv(BINDINGDB_FILE, sep="\t", dtype=str, on_bad_lines='skip')
print(f"Loaded {len(df):,} rows")


Loading BindingDB TSV...
Loaded 3,078,835 rows


# Find and select all relevant columns

In [11]:
uniprot_cols = [col for col in df.columns if 'UniProt (SwissProt) Primary ID of Target Chain' in col]
print(f"Found {len(uniprot_cols)} UniProt target chain columns")

required_columns = ["Ligand SMILES", "Ki (nM)", "IC50 (nM)", "Kd (nM)", "EC50 (nM)"] + uniprot_cols
df = df[required_columns].copy()

Found 50 UniProt target chain columns


# Parse affinity values

In [12]:
def parse_affinity(x):
    if pd.isna(x):
        return np.nan
    try:
        s = str(x).strip()
        if s.startswith("<"):
            return float(s[1:])
        elif s.startswith(">"):
            return 999999
        return float(s)
    except:
        return np.nan

for col in tqdm(["Ki (nM)", "IC50 (nM)", "Kd (nM)", "EC50 (nM)"], desc="Parsing affinities"):
    df[col + "_num"] = df[col].apply(parse_affinity)

df["min_affinity"] = df[["Ki (nM)_num", "IC50 (nM)_num", "Kd (nM)_num", "EC50 (nM)_num"]].min(axis=1, skipna=True)

Parsing affinities: 100%|██████████| 4/4 [00:10<00:00,  2.58s/it]


# Filter to bioactive compounds

In [13]:
df_bioactive = df[df["min_affinity"] <= AFFINITY_CUTOFF].copy()
print(f"\nBioactive compounds (≤{AFFINITY_CUTOFF} nM): {len(df_bioactive):,} rows")


Bioactive compounds (≤1000 nM): 1,907,542 rows


# Check if any target is a kinase

In [14]:
df_bioactive['is_kinase_inhibitor'] = False

for col in tqdm(uniprot_cols, desc="Checking chains"):
    df_bioactive['is_kinase_inhibitor'] |= df_bioactive[col].isin(kinase_ids)

n_kinase_rows = df_bioactive['is_kinase_inhibitor'].sum()
print(f"Rows with kinase target: {n_kinase_rows:,}")
print(f"Rows with non-kinase target: {len(df_bioactive) - n_kinase_rows:,}")

Checking chains: 100%|██████████| 50/50 [00:03<00:00, 16.07it/s]

Rows with kinase target: 546,101
Rows with non-kinase target: 1,361,441


# Standardize SMILES and generate InChI Keys

In [15]:
uncharger = rdMolStandardize.Uncharger()
def standardize_molecule(smi, uncharger):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None, None
    
        mol_std = rdMolStandardize.ChargeParent(mol)
        if mol_std is None:
            return None, None
        
        mol_uncharged = uncharger.uncharge(mol_std)
        smiles_std = Chem.MolToSmiles(mol_uncharged)
        inchi_key = Chem.MolToInchiKey(mol_uncharged)
        return smiles_std, inchi_key
    except:
        return None, None

unique_smiles = df_bioactive["Ligand SMILES"].dropna().unique()

smiles_map = {}
inchi_map = {}

for smi in tqdm(unique_smiles, desc="Standardizing"):
    smiles_std, inchi_key = standardize_molecule(smi, uncharger)
    if smiles_std and inchi_key:
        smiles_map[smi] = smiles_std
        inchi_map[smi] = inchi_key

df_bioactive['SMILES'] = df_bioactive['Ligand SMILES'].map(smiles_map)
df_bioactive['InChI_Key'] = df_bioactive['Ligand SMILES'].map(inchi_map)
df_bioactive = df_bioactive.dropna(subset=['SMILES', 'InChI_Key'])

print(f"Standardized {len(smiles_map):,}/{len(unique_smiles):,} unique SMILES")

Standardizing: 100%|██████████| 978431/978431 [1:03:42<00:00, 255.95it/s]


Standardized 976,504/978,431 unique SMILES


# Separate kinase and non-kinase inhibitors

In [16]:
df_kinase = df_bioactive[df_bioactive['is_kinase_inhibitor']].copy()
df_nonkinase = df_bioactive[~df_bioactive['is_kinase_inhibitor']].copy()

print(f"\nKinase inhibitor rows: {len(df_kinase):,}")
print(f"Non-kinase inhibitor rows: {len(df_nonkinase):,}")


Kinase inhibitor rows: 545,679
Non-kinase inhibitor rows: 1,358,400


# Deduplicate kinase set by InChI Key

In [17]:
kinase_inchi_to_smiles = {}
for smi, inchi in zip(df_kinase['SMILES'], df_kinase['InChI_Key']):
    if inchi not in kinase_inchi_to_smiles:
        kinase_inchi_to_smiles[inchi] = smi

kinase_unique_inchi = set(kinase_inchi_to_smiles.keys())
print(f"Unique kinase compounds: {len(kinase_unique_inchi):,}")

Unique kinase compounds: 252,595


# Remove overlap from non-kinase set

In [18]:
df_nonkinase = df_nonkinase[~df_nonkinase['InChI_Key'].isin(kinase_unique_inchi)]
print(f"Non-kinase rows after overlap removal: {len(df_nonkinase):,}")

Non-kinase rows after overlap removal: 1,337,056


# Deduplicate non-kinase set by InChI Key

In [19]:
nonkinase_inchi_to_smiles = {}
for smi, inchi in zip(df_nonkinase['SMILES'], df_nonkinase['InChI_Key']):
    if inchi not in nonkinase_inchi_to_smiles:
        nonkinase_inchi_to_smiles[inchi] = smi

nonkinase_unique_inchi = set(nonkinase_inchi_to_smiles.keys())
print(f"Unique non-kinase compounds: {len(nonkinase_unique_inchi):,}")

Unique non-kinase compounds: 711,481


# Get final SMILES lists

In [20]:
final_kinase_smiles = list(kinase_inchi_to_smiles.values())
final_nonkinase_smiles = list(nonkinase_inchi_to_smiles.values())

print(f"\nFinal unique compounds:")
print(f"  Kinase inhibitors: {len(final_kinase_smiles):,}")
print(f"  Non-kinase inhibitors: {len(final_nonkinase_smiles):,}")


Final unique compounds:
  Kinase inhibitors: 252,595
  Non-kinase inhibitors: 711,481


# Save outputs

In [21]:
kinase_output = pd.DataFrame({'SMILES': sorted(final_kinase_smiles)})
kinase_output.to_csv(OUTPUT_KINASE, index=False)
print(f"\nSaved: {OUTPUT_KINASE}")

nonkinase_output = pd.DataFrame({'SMILES': sorted(final_nonkinase_smiles)})
nonkinase_output.to_csv(OUTPUT_NON_KINASE, index=False)
print(f"Saved: {OUTPUT_NON_KINASE}")


Saved: c:\Users\bstrickland\Documents\Projects\2026_KIPS_Final_Revisions\Code\Output\Intermediate\Regenerated_Inputs\BindingDB_Kinase_Inhibitors.csv
Saved: c:\Users\bstrickland\Documents\Projects\2026_KIPS_Final_Revisions\Code\Output\Intermediate\Regenerated_Inputs\BindingDB_Non_Kinase_Inhibitors.csv
